# 卷积神经网络入门

这个文件原先是空的。这里补一个可以在 CPU 上直接跑的小例子：
用两层卷积区分合成的 28×28 灰度图，不下载数据集。
若要换成 Fashion-MNIST，把 `make_loader` 换成 `torchvision.datasets.FashionMNIST` 即可，网络结构不用改。

In [ ]:
import torch
from torch import nn

torch.manual_seed(0)

class SmallCNN(nn.Module):
    """两层卷积加全连接，输入形状是 (N, 1, 28, 28)。"""

    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 6, kernel_size=5),  # 28 -> 24
            nn.ReLU(),
            nn.MaxPool2d(2),                 # 24 -> 12
            nn.Conv2d(6, 16, kernel_size=5), # 12 -> 8
            nn.ReLU(),
            nn.MaxPool2d(2),                 # 8 -> 4
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 4 * 4, 64),
            nn.ReLU(),
            nn.Linear(64, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

def make_batch(batch_size=64):
    """按类别在图像中心放不同亮度的方块，让卷积有可见的空间模式。"""
    labels = torch.randint(0, 10, (batch_size,))
    images = torch.randn(batch_size, 1, 28, 28) * 0.1
    for i, label in enumerate(labels):
        start = int(label) + 4
        images[i, 0, start:start + 6, 10:16] += 1.5
    return images, labels

def train(epochs=3, steps_per_epoch=20):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    net = SmallCNN().to(device)
    optimizer = torch.optim.SGD(net.parameters(), lr=0.1)
    loss_fn = nn.CrossEntropyLoss()
    for epoch in range(epochs):
        total = 0.0
        for _ in range(steps_per_epoch):
            X, y = make_batch()
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            loss = loss_fn(net(X), y)
            loss.backward()
            optimizer.step()
            total += float(loss.detach())
        print(f"epoch {epoch + 1}, loss {total / steps_per_epoch:.4f}")
    return net

model = train()
model.eval()
with torch.no_grad():
    X, y = make_batch(128)
    pred = model(X).argmax(dim=1)
    acc = float((pred == y).float().mean())
print(f"holdout accuracy {acc:.3f}")